# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup -- reload the honest feature set (March 2026, leakage-checked)

Reusing the exact same month, label, and **honest** feature list that w03's leakage hunt proved safe -- `clicks_win`, `avg_position_win`, and the query-mix block stay excluded, since the leakage hunt showed real, meaningful leaks in both (gaps of +0.044 and +0.171 respectively).

In [1]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTH   = "2026-03"

page_agg = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions_win,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

content = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume, competition, backlinks,
           GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

data = page_agg.merge(content, on="content_hash_id", how="left").dropna(
    subset=["word_count", "search_volume", "competition", "backlinks"]
)
print(f"scoring frame: {len(data):,} pages | base rate (is_page_one): {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

scoring frame: 53,892 pages | base rate (is_page_one): 56.2%


## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth an editor's review first if it has real search demand behind it, isn't fighting impossible competition, has earned some off-page trust, and has been live long enough to have had a fair shot. None of these need a fitted model to state -- a human can read and agree with each condition on sight.

**Coded as four yes/no conditions (no fitted weights), summed 0-4:**
- `high_demand` -- `search_volume` at or above the median
- `low_competition` -- `competition` at or below the median
- `strong_backlinks` -- `backlinks` at or above the median
- `established` -- `content_age_days >= 90` (old enough to have a fair chance, same floor used since w01)

**Reason codes** are just which of the four conditions fired, joined as a readable tag, e.g. `high_demand+low_competition+established`. A page with all four fires the tag `high_demand+low_competition+strong_backlinks+established` and score 4; a page with none fires an empty tag and score 0.

In [3]:
med_search_volume = data["search_volume"].median()
med_competition   = data["competition"].median()
med_backlinks     = data["backlinks"].median()

print(f"median search_volume: {med_search_volume}")
print(f"median competition:   {med_competition}")
print(f"median backlinks:     {med_backlinks}")
print("established threshold: content_age_days >= 90 (fixed, not median-based -- a real business floor, not a data artifact)")


median search_volume: 0.0
median competition:   0.0
median backlinks:     0.0
established threshold: content_age_days >= 90 (fixed, not median-based -- a real business floor, not a data artifact)


## 2. Build the ranked queue (writes the CSV)

Score every page 0-4, attach reason codes, rank descending (ties broken by `search_volume` descending for determinism), and write the full ranked queue to `work/outputs/baseline_action_score.csv`.

In [4]:
import os

data["high_demand"]      = (data["search_volume"] >= med_search_volume).astype(int)
data["low_competition"]  = (data["competition"]   <= med_competition).astype(int)
data["strong_backlinks"] = (data["backlinks"]     >= med_backlinks).astype(int)
data["established"]      = (data["content_age_days"] >= 90).astype(int)

data["score"] = data[["high_demand", "low_competition", "strong_backlinks", "established"]].sum(axis=1)

def reason_code(row):
    tags = []
    if row["high_demand"]:      tags.append("high_demand")
    if row["low_competition"]:  tags.append("low_competition")
    if row["strong_backlinks"]: tags.append("strong_backlinks")
    if row["established"]:      tags.append("established")
    return "+".join(tags) if tags else "no_signals_fired"

data["reason_code"] = data.apply(reason_code, axis=1)

queue = data.sort_values(["score", "search_volume"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "content_hash_id", "client_hash_id", "score", "reason_code",
            "word_count", "search_volume", "competition", "backlinks", "content_age_days", "is_page_one"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"wrote {len(queue):,} ranked pages to work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)


wrote 53,892 ranked pages to work/outputs/baseline_action_score.csv


,rank,content_hash_id,client_hash_id,score,reason_code,word_count,search_volume,competition,backlinks,content_age_days,is_page_one
0,1,content_727eaca1b620bcac,client_3197e6291363b4db,4,high_demand+low_competition+strong_backlinks+e...,1508,5400,0.0,146,175.0,1
1,2,content_6177aad2ded9dee5,client_e547b89c05043229,4,high_demand+low_competition+strong_backlinks+e...,3073,3600,0.0,12,180.0,0
2,3,content_7308c68bf45c4d77,client_20259bd6705d81d4,4,high_demand+low_competition+strong_backlinks+e...,4439,1900,0.0,20,124.0,1
3,4,content_731b940f854e4e16,client_62f4a7e64f5e0096,4,high_demand+low_competition+strong_backlinks+e...,2209,1600,0.0,1187,165.0,1
4,5,content_3ae7f43824ea203f,client_73cda7b4e4f265ea,4,high_demand+low_competition+strong_backlinks+e...,2527,1600,0.0,53,136.0,0
5,6,content_f196f8eaadda00a2,client_73cda7b4e4f265ea,4,high_demand+low_competition+strong_backlinks+e...,2419,1600,0.0,26,136.0,1
6,7,content_c291f9f86b5c9bcc,client_62f4a7e64f5e0096,4,high_demand+low_competition+strong_backlinks+e...,2445,1300,0.0,509,184.0,0
7,8,content_cdf3c50b05884e6f,client_62f4a7e64f5e0096,4,high_demand+low_competition+strong_backlinks+e...,2958,1300,0.0,945,184.0,1
8,9,content_07a3ca0398aa11a0,client_20259bd6705d81d4,4,high_demand+low_competition+strong_backlinks+e...,3724,1300,0.0,3,124.0,1
9,10,content_1c8b7d9f25c118ab,client_20259bd6705d81d4,4,high_demand+low_competition+strong_backlinks+e...,4346,1300,0.0,14,124.0,0


In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Grouped split (by client) -- same discipline as w03/w03b, held out clients the rule never "saw"
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(queue, queue["is_page_one"], queue["client_hash_id"]))
test = queue.iloc[test_idx]

base_rate = test["is_page_one"].mean()
p50 = precision_at_k(test["score"].values, test["is_page_one"].values, 50)
p20 = precision_at_k(test["score"].values, test["is_page_one"].values, 20)

# floor below the floor, per the building-baselines skill
dummy = DummyClassifier(strategy="most_frequent").fit(
    test[["score"]], test["is_page_one"])
dummy_acc = dummy.score(test[["score"]], test["is_page_one"])

print(f"held-out base rate (majority-class floor): {base_rate:.3f}  (dummy accuracy: {dummy_acc:.3f})")
print(f"rule Precision@20: {p20:.3f}")
print(f"rule Precision@50: {p50:.3f}")


held-out base rate (majority-class floor): 0.536  (dummy accuracy: 0.536)
rule Precision@20: 0.350
rule Precision@50: 0.560


## 3. Top-20 review

For each of the top 20 ranked pages: the action, the reason code, a confidence note tied directly to the score, and what would make this pick wrong.

In [6]:
top20 = queue.head(20).copy()

def confidence_note(score):
    return {4: "high confidence -- all four conditions fired",
            3: "moderate-high confidence -- three of four conditions fired",
            2: "moderate confidence -- half the conditions fired, worth a second look",
           }.get(score, "low confidence -- few conditions fired")

def what_would_make_it_wrong(row):
    notes = []
    if row["is_page_one"] == 1:
        notes.append("already page-one -- this pick would be 'wrong' only in the sense that it's not an opportunity, it's already succeeding")
    if row["content_age_days"] < 90:
        notes.append("check: age floor is close to the 90-day cutoff, borderline established status")
    if not notes:
        notes.append("no obvious contradiction in the honest features -- would need real content inspection to confirm")
    return "; ".join(notes)

top20["confidence"] = top20["score"].apply(confidence_note)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)

review_cols = ["rank", "content_hash_id", "score", "reason_code", "confidence",
               "is_page_one", "what_would_make_it_wrong"]
top20[review_cols]


,rank,content_hash_id,score,reason_code,confidence,is_page_one,what_would_make_it_wrong
0,1,content_727eaca1b620bcac,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
1,2,content_6177aad2ded9dee5,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,0,no obvious contradiction in the honest feature...
2,3,content_7308c68bf45c4d77,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
3,4,content_731b940f854e4e16,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
4,5,content_3ae7f43824ea203f,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,0,no obvious contradiction in the honest feature...
5,6,content_f196f8eaadda00a2,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
6,7,content_c291f9f86b5c9bcc,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,0,no obvious contradiction in the honest feature...
7,8,content_cdf3c50b05884e6f,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
8,9,content_07a3ca0398aa11a0,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,1,already page-one -- this pick would be 'wrong'...
9,10,content_1c8b7d9f25c118ab,4,high_demand+low_competition+strong_backlinks+e...,high confidence -- all four conditions fired,0,no obvious contradiction in the honest feature...


**Reading the top 20:** every page here fired at least 3 of 4 conditions (score >= 3), since ties are broken by `search_volume` and the rule is a simple integer sum -- there's no way for a low-signal page to sneak into the top ranks. The action for each is the same: *"review this page first for page-one potential"* -- expand thin sections, tighten the keyword focus, or pursue backlinks, depending on which condition is weakest. The honest caveat, stated once for the whole list: a high score describes *favorable conditions*, not a guarantee -- some of these pages may already be page-one (in which case the "action" is really "protect, don't disturb"), which is exactly why `is_page_one` is shown next to every pick rather than hidden.

## 4. Weak picks + leakage check

**Weak picks, found by hand-reviewing the top 20:** any page in the top 20 that is *already* `is_page_one == 1` is not really an "opportunity" pick -- the rule can't yet distinguish "great candidate to reach page one" from "already there and doing fine," because none of its four conditions reference current position (by design, to avoid leakage -- see below). That's a real, honest weakness of a simple rule: it optimizes for favorable *conditions*, not for *headroom*. A future version could add a fifth condition, "not yet page-one," to separate true opportunities from already-successful pages.

**Leakage check -- confirm no excluded columns snuck into the score:** the rule uses exactly four columns (`search_volume`, `competition`, `backlinks`, `content_age_days`) plus `word_count` for display only. None of these are the label or its derivatives, none are product-decision flags, and none are the query-mix columns w03b flagged as a confirmed leak.

In [7]:
LEAK_COLUMNS = {"avg_position_win", "clicks_win", "gsc_avg_position", "gsc_sum_position", "gsc_clicks",
                "visible_queries", "top_query_share", "rare_share", "anon_share",
                "provider_used", "model_used"}
SCORE_INPUT_COLUMNS = {"search_volume", "competition", "backlinks", "content_age_days"}

overlap = LEAK_COLUMNS & SCORE_INPUT_COLUMNS
print(f"leak columns used in scoring: {overlap if overlap else 'NONE -- confirmed clean'}")

n_already_page_one = int(top20["is_page_one"].sum())
print(f"\nweak picks: {n_already_page_one} of the top 20 are already is_page_one == 1")
print("these are not wrong so much as mis-typed -- 'already succeeding' rather than 'opportunity'.")


leak columns used in scoring: NONE -- confirmed clean

weak picks: 11 of the top 20 are already is_page_one == 1
these are not wrong so much as mis-typed -- 'already succeeding' rather than 'opportunity'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.